# Lab 05: The Verification Bottleneck

Chapter 5 warns that flooding a narrow, expensive verification pipeline with cheap AI-generated candidates will fail *unless* the system uses structural proxies to prune them early.

## Exercise: Architecting Around the Bottleneck
We will simulate generating 1,000 AI candidates. We implement a fast, imperfect structural proxy to filter out the obvious failures *before* they hit the expensive verification simulator. This demonstrates how to solve the O(N) verification bottleneck.

In [ ]:
import time
import random
import matplotlib.pyplot as plt

#@title AI Generation Scale
candidate_count = 500 #@param {type:"slider", min:100, max:2000, step:100}
use_structural_proxy = True #@param {type:"boolean"}

def generate_candidates(n):
    return [{'id': i, 'heuristic_score': random.random()} for i in range(n)]

def fast_structural_proxy(candidate):
    # Filters out candidates with bad heuristics instantly
    return candidate['heuristic_score'] > 0.6

def expensive_verification(candidate):
    time.sleep(0.005) # Expensive simulator
    return random.random() > 0.5

candidates = generate_candidates(candidate_count)
start = time.time()

passed = []
if use_structural_proxy:
    # 1. Prune with Proxy
    pruned = [c for c in candidates if fast_structural_proxy(c)]
    # 2. Run Expensive Verification only on survivors
    passed = [c for c in pruned if expensive_verification(c)]
else:
    passed = [c for c in candidates if expensive_verification(c)]

duration = time.time() - start
print(f"Verification took {duration:.2f} seconds.")
print(f"Passed candidates: {len(passed)} / {len(candidates)}")


### Reflection
Notice how the `use_structural_proxy` toggle drastically reduces the runtime for large candidate pools. This is the only way AI generation scales!